# Artifact Evaluation — Fine-Grained Soundscape Control for Augmented Hearing

**Paper**: MobiSys 2026 #198 | [arXiv:2603.00395](https://arxiv.org/abs/2603.00395)

This notebook reproduces the key results from the paper. Two evaluation modes are available:

| Mode | Models | Time | Coverage |
|------|--------|------|----------|
| **Quick** | 3 TSE + 2 SED | ~1.5 hours | Tables 1, 3, 4 (subset) |
| **Full** | 12 TSE + 7 SED | ~10 hours | Tables 1, 2, 3, 4 (complete) |

---

## 0. Environment Check

In [ ]:
import torch, os
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")
print(f"Dataset: {len(os.listdir('/data/BinauralCuratedDataset/'))} dirs")
print("Ready!")

## 1. Quick Evaluation (~1.5 hours)

Evaluates a representative subset of the paper results:
- **TSE (Table 1 & 3)**: Orange Pi, Raspberry Pi, NeuralAids — all with FiLM=All, 2000 samples
- **SED (Table 4)**: Best case (1 target, 1 background) + Worst case (5 targets, 1-3 backgrounds)

In [ ]:
!cd /app && bash scripts/eval/eval_quick.sh /data /output/quick

### Quick Eval — Expected Results

**TSE (FiLM=All, 5 output channels, 1-5 targets in mixture)**:

| Model | SNRi (expected) | SI-SDRi (expected) | Paper SNRi | Paper SI-SDRi |
|-------|----------------|-------------------|-----------|---------------|
| Orange Pi | ~12.3 | ~10.1 | 12.26 | 10.16 |
| Raspberry Pi | ~10.5 | ~8.0 | — | — |
| NeuralAids | ~10.6 | ~7.4 | 10.50 | 7.27 |

**SED (Fine-tuned AST, 2000 samples)**:

| Condition | Acc (expected) | F1 (expected) | Paper Acc | Paper F1 |
|-----------|---------------|--------------|----------|----------|
| tgt=1 (best) | ~0.991 | ~0.914 | 0.992 | 0.921 |
| tgt=5 (worst) | ~0.918 | ~0.827 | — | — |

---

## 2. Full Evaluation (~10 hours)

Reproduces all paper tables:
- **Table 1**: 4 TSE models (Orange Pi, Raspberry Pi, NeuralAids, Waveformer)
- **Table 2**: Multi-output (5-out, 20-out)
- **Table 3**: FiLM ablation (6 variants)
- **Table 4**: SED (7 conditions)

In [ ]:
# Table 1: TSE Model Comparison
!cd /app && bash scripts/eval/run_tse.sh /data /output/full/table1

In [ ]:
# Table 2: Multi-output TSE
!cd /app && bash scripts/eval/run_multiout.sh /data /output/full/table2

In [ ]:
# Table 3: FiLM Ablation
!cd /app && bash scripts/eval/run_ablation.sh /data /output/full/table3

In [ ]:
# Table 4: SED Evaluation
!cd /app && bash scripts/eval/run_sed.sh /data /output/full/table4

### Full Eval — Expected Results

**Table 1: TSE Model Comparison** (single source, single output channel)

| Model | Params | SNRi (paper) | SI-SNRi (paper) |
|-------|--------|--------------|-----------------|
| Orange Pi | 0.498M | 11.99 ± 7.04 | 11.27 ± 8.21 |
| Raspberry Pi | 0.208M | 10.13 ± 4.01 | 7.72 ± 5.17 |
| NeuralAids | 0.502M | 9.75 ± 4.80 | 7.60 ± 6.48 |
| Waveformer | 1.207M | 7.29 ± 6.11 | 5.58 ± 7.67 |

**Table 3: FiLM Ablation** (5 output channels, 1-5 targets)

| Model | FiLM | SNRi (paper) | SI-SNRi (paper) |
|-------|------|--------------|-----------------|
| Orange Pi | First only | 11.76 ± 4.27 | 9.51 ± 5.43 |
| Orange Pi | **All blocks** | **12.26 ± 4.38** | **10.16 ± 5.72** |
| Orange Pi | All except first | 11.88 ± 4.27 | 9.77 ± 5.55 |
| NeuralAids | First only | 9.46 ± 3.81 | 5.73 ± 6.21 |
| NeuralAids | **All blocks** | **10.50 ± 4.15** | **7.27 ± 6.28** |
| NeuralAids | All except first | 10.19 ± 4.01 | 7.09 ± 5.38 |

**Table 4: SED** (Fine-tuned AST, per-class F1-maximizing thresholds)

| #Targets | #Backgrounds | Accuracy (paper) | F1 (paper) |
|----------|-------------|-----------------|------------|
| 1 | 1 | 0.992 | 0.921 |
| 2 | 1 | 0.971 | 0.845 |
| 3 | 1 | 0.958 | 0.851 |
| 4 | 1 | 0.943 | 0.849 |
| 5 | 1 | 0.923 | 0.838 |
| 1 | 1-3 | — | — |
| 5 | 1-3 | — | — |

---

## 3. View Results

After evaluation completes, view the results:

In [ ]:
import json, os, glob

def show_results(base_dir):
    # TSE results
    tse_files = sorted(glob.glob(f"{base_dir}/**/metrics_total_averages.json", recursive=True))
    if tse_files:
        print("=" * 60)
        print("TSE Results")
        print("=" * 60)
        print(f"{'Model':<35} {'SNRi':>8} {'SI-SDRi':>8}")
        print("-" * 55)
        for f in tse_files:
            name = f.split(base_dir)[-1].strip('/')
            name = '/'.join(name.split('/')[:-1])  # remove filename
            with open(f) as fp:
                d = json.load(fp)
            print(f"{name:<35} {d['snr_i']:>8.2f} {d['si_sdr_i']:>8.2f}")

    # SED results
    sed_files = sorted(glob.glob(f"{base_dir}/**/metrics.json", recursive=True))
    if sed_files:
        print()
        print("=" * 60)
        print("SED Results")
        print("=" * 60)
        print(f"{'Condition':<25} {'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6}")
        print("-" * 55)
        for f in sed_files:
            name = f.split(base_dir)[-1].strip('/')
            name = '/'.join(name.split('/')[:-1])
            with open(f) as fp:
                d = json.load(fp)
            if 'accuracy' in d:
                print(f"{name:<25} {d['accuracy']:>6.3f} {d['precision']:>6.3f} {d['recall']:>6.3f} {d['f1']:>6.3f}")

# Show quick eval results
if os.path.exists('/output/quick'):
    print("\n*** Quick Eval Results ***\n")
    show_results('/output/quick')

# Show full eval results
if os.path.exists('/output/full'):
    print("\n*** Full Eval Results ***\n")
    show_results('/output/full')